# Isolation Forest ve LOF ile Credit Card Fraud Tespiti

Bu notebook, Credit Card veri seti üzerinde Isolation Forest ve Local Outlier Factor modellerini eğitir.

## Modeller:
- **Isolation Forest**: Ağaç tabanlı anomali tespiti
- **Local Outlier Factor (LOF)**: Komşuluk tabanlı anomali tespiti

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

print("Kütüphaneler yüklendi.")

Kütüphaneler yüklendi.


## 1. Credit Card Verisini Yükle

In [2]:
# Credit Card verisini yükle
cc_df = pd.read_csv("../data/credit_card_PCA.csv")
print(f"Veri seti yüklendi: {cc_df.shape}")

# Feature ve target ayır
X_cc = cc_df.drop(columns=['Class']).values.astype(np.float32)
y_cc = cc_df['Class'].values

print(f"Sınıf dağılımı: Normal={sum(y_cc==0)}, Fraud={sum(y_cc==1)}")

Veri seti yüklendi: (284807, 30)
Sınıf dağılımı: Normal=284315, Fraud=492


## 2. Çapraz Doğrulama Ayarları

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
repeats = 3

results_iso_lof = {
    'IsolationForest': [],
    'LOF': []
}

print(f"K-Fold: 5, Tekrar: {repeats}")

K-Fold: 5, Tekrar: 3


## 3. Model Eğitimi

In [4]:
for r in range(repeats):
    print(f"\n--- Tekrar {r+1}/{repeats} ---")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + r)
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_cc, y_cc)):
        X_train, X_test = X_cc[train_idx], X_cc[test_idx]
        y_train, y_test = y_cc[train_idx], y_cc[test_idx]
        
        # Majority class (normal) eğitim verisi
        X_train_major = X_train[y_train == 0]
        
        # -- Isolation Forest --
        iso = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
        iso.fit(X_train_major)
        iso_scores = -iso.decision_function(X_test)
        
        auc_iso = roc_auc_score(y_test, iso_scores)
        aupr_iso = average_precision_score(y_test, iso_scores)
        results_iso_lof['IsolationForest'].append((auc_iso, aupr_iso))
        
        # -- LOF --
        lof = LocalOutlierFactor(n_neighbors=20, contamination=0.01, novelty=True, n_jobs=-1)
        lof.fit(X_train_major)
        lof_scores = -lof.decision_function(X_test)
        
        auc_lof = roc_auc_score(y_test, lof_scores)
        aupr_lof = average_precision_score(y_test, lof_scores)
        results_iso_lof['LOF'].append((auc_lof, aupr_lof))
        
        if fold == 1:
            print(f"  Fold 1 - IsoForest: AUC={auc_iso:.4f}, LOF: AUC={auc_lof:.4f}")

print("\n✓ Eğitim tamamlandı!")


--- Tekrar 1/3 ---
  Fold 1 - IsoForest: AUC=0.9560, LOF: AUC=0.6704

--- Tekrar 2/3 ---
  Fold 1 - IsoForest: AUC=0.9514, LOF: AUC=0.6929

--- Tekrar 3/3 ---
  Fold 1 - IsoForest: AUC=0.9521, LOF: AUC=0.6225

✓ Eğitim tamamlandı!


## 4. Değerlendirme

In [5]:
print("="*60)
print("CREDIT CARD - ISOLATION FOREST & LOF SONUÇLARI")
print("="*60)

print(f"{'Model':20s} | {'Mean AUC':10s} | {'Mean AUPRC':10s}")
print("-"*45)

for method in results_iso_lof:
    scores = results_iso_lof[method]
    if scores:
        arr = np.array(scores)
        mean_auc, mean_aupr = arr.mean(axis=0)
        print(f'{method:20s} | {mean_auc:.4f}     | {mean_aupr:.4f}')

CREDIT CARD - ISOLATION FOREST & LOF SONUÇLARI
Model                | Mean AUC   | Mean AUPRC
---------------------------------------------
IsolationForest      | 0.9521     | 0.1220
LOF                  | 0.6425     | 0.0039


## 5. Sonuçları Kaydet

In [6]:
os.makedirs("../pickled_storage", exist_ok=True)

with open("../pickled_storage/results_cc_iso_lof.pkl", "wb") as f:
    pickle.dump(results_iso_lof, f)

print("✓ Sonuçlar kaydedildi: pickled_storage/results_cc_iso_lof.pkl")

✓ Sonuçlar kaydedildi: pickled_storage/results_cc_iso_lof.pkl
